In [88]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

### Global variables

In [89]:
symbol = "USDJPY"
timeframe = "PERIOD_D1"
lookahead = 1
common_path = r"C:\Users\Omega Joctan\AppData\Roaming\MetaQuotes\Terminal\Common\Files"

### Saving model to ONNX

In [90]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import os

def saveModel(model, n_features: int, technique_name: str):
    
    initial_type = [("input", FloatTensorType([None, n_features]))]
    
    onnx_model = convert_sklearn(model, initial_types=initial_type, target_opset=14)

    with open(os.path.join(common_path, f"{symbol}.{timeframe}.{technique_name}.onnx"), "wb") as f:
        f.write(onnx_model.SerializeToString())

In [91]:
symbols = [
    "EURUSD",
    "USTEC",
    "XAUUSD",
    "USDJPY",
    "BTCUSD",
    "CA60",
    "UK100"
]

for s in symbols:
    
    df = pd.read_csv(f"{common_path}\{s}.{timeframe}.data.csv")
    df["Candle type"] = (df["Close"] > df["Open"]).astype(int)
    
    print(f"{s}(unique):",np.unique(df["Candle type"], return_counts=True))

EURUSD(unique): (array([0, 1]), array([252, 267]))
USTEC(unique): (array([0, 1]), array([231, 285]))
XAUUSD(unique): (array([0, 1]), array([246, 270]))
USDJPY(unique): (array([0, 1]), array([208, 311]))
BTCUSD(unique): (array([0, 1]), array([332, 400]))
CA60(unique): (array([0, 1]), array([230, 271]))
UK100(unique): (array([0, 1]), array([239, 266]))


In [92]:
df = pd.read_csv(f"{common_path}\{symbol}.{timeframe}.data.csv")

df["future_close"] = df["Close"].shift(-lookahead) # future closing price based on lookahead value
df.dropna(inplace=True)

df["Signal"] = (df["future_close"] > df["Close"]).astype(int)

print("Signals(unique): ",np.unique(df["Signal"], return_counts=True))

Signals(unique):  (array([0, 1]), array([225, 293]))


In [93]:
df

,Open,High,Low,Close,future_close,Signal
0,130.924,131.046,130.601,130.601,130.985,1
1,130.698,131.400,129.517,130.985,132.652,1
2,131.014,132.716,129.926,132.652,133.421,1
3,132.570,134.055,131.684,133.421,132.106,0
4,133.406,134.774,131.998,132.106,131.888,0
...,...,...,...,...,...,...
513,156.298,157.270,156.288,157.163,157.172,1
514,157.102,157.394,156.891,157.172,158.017,1
515,157.156,158.085,157.075,158.017,157.790,0
516,157.872,157.956,157.352,157.790,156.853,0


In [94]:
X = df.drop(columns=["Signal", "future_close"])
y = df["Signal"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, shuffle=False)

classes present in the data

In [95]:
print("classes in y: ",np.unique(y, return_counts=True))

classes in y:  (array([0, 1]), array([225, 293]))


### No-Sampling

In [96]:
model = RandomForestClassifier(n_estimators=100,
                               max_depth=5,
                               min_samples_split=3,
                               random_state=42)

model.fit(X_train, y_train)

RandomForestClassifier(max_depth=5, min_samples_split=3, random_state=42)

In [97]:
saveModel(model=model, n_features=X_train.shape[1], technique_name="no-sampling")

### Model evaluation

In [98]:
y_train_pred = model.predict(X_train)

print("Train Classification report\n",classification_report(y_train, y_train_pred))

y_test_pred = model.predict(X_test)

print("Test Classification report\n",classification_report(y_test, y_test_pred))

Train Classification report
               precision    recall  f1-score   support

           0       0.98      0.41      0.57       158
           1       0.68      1.00      0.81       204

    accuracy                           0.74       362
   macro avg       0.83      0.70      0.69       362
weighted avg       0.81      0.74      0.71       362

Test Classification report
               precision    recall  f1-score   support

           0       0.49      0.27      0.35        67
           1       0.59      0.79      0.67        89

    accuracy                           0.56       156
   macro avg       0.54      0.53      0.51       156
weighted avg       0.54      0.56      0.53       156



### Random Oversampling

In [99]:
from imblearn.over_sampling import RandomOverSampler

print("b4 Target: ",np.unique(y_train, return_counts=True))

rus = RandomOverSampler(random_state=42)
X_resampled, y_resampled = rus.fit_resample(X_train, y_train)

print("After Target: ",np.unique(y_resampled, return_counts=True))

b4 Target:  (array([0, 1]), array([158, 204]))
After Target:  (array([0, 1]), array([204, 204]))


In [100]:
model.fit(X_resampled, y_resampled)

RandomForestClassifier(max_depth=5, min_samples_split=3, random_state=42)

In [101]:
saveModel(model=model, n_features=X_train.shape[1], technique_name="randomoversampling")

### Model evaluation

In [102]:
y_train_pred = model.predict(X_train)

print("Train Classification report\n",classification_report(y_train, y_train_pred))

y_test_pred = model.predict(X_test)

print("Test Classification report\n",classification_report(y_test, y_test_pred))

Train Classification report
               precision    recall  f1-score   support

           0       0.82      0.85      0.83       158
           1       0.88      0.86      0.87       204

    accuracy                           0.85       362
   macro avg       0.85      0.85      0.85       362
weighted avg       0.85      0.85      0.85       362

Test Classification report
               precision    recall  f1-score   support

           0       0.39      0.43      0.41        67
           1       0.54      0.49      0.51        89

    accuracy                           0.47       156
   macro avg       0.46      0.46      0.46       156
weighted avg       0.47      0.47      0.47       156



### Random undersampling

In [103]:
from imblearn.under_sampling import RandomUnderSampler

print("b4 Target: ",np.unique(y_train, return_counts=True))

rus = RandomUnderSampler(random_state=42)
X_resampled, y_resampled = rus.fit_resample(X_train, y_train)

print("After Target: ",np.unique(y_resampled, return_counts=True))

b4 Target:  (array([0, 1]), array([158, 204]))
After Target:  (array([0, 1]), array([158, 158]))


In [104]:
model.fit(X_resampled, y_resampled)

RandomForestClassifier(max_depth=5, min_samples_split=3, random_state=42)

In [105]:
saveModel(model=model, n_features=X_train.shape[1], technique_name="randomundersampling")

In [106]:
y_train_pred = model.predict(X_train)

print("Train Classification report\n",classification_report(y_train, y_train_pred))

y_test_pred = model.predict(X_test)

print("Test Classification report\n",classification_report(y_test, y_test_pred))

Train Classification report
               precision    recall  f1-score   support

           0       0.76      0.90      0.82       158
           1       0.91      0.78      0.84       204

    accuracy                           0.83       362
   macro avg       0.83      0.84      0.83       362
weighted avg       0.84      0.83      0.83       362

Test Classification report
               precision    recall  f1-score   support

           0       0.41      0.58      0.48        67
           1       0.55      0.38      0.45        89

    accuracy                           0.47       156
   macro avg       0.48      0.48      0.47       156
weighted avg       0.49      0.47      0.46       156



### Tomek Links

In [107]:
from imblearn.under_sampling import TomekLinks

tl = TomekLinks()
X_resampled, y_resampled = tl.fit_resample(X_train, y_train)

print(f"Before --> y (unique): {np.unique(y_train, return_counts=True)}\nAfter  --> y (unique): {np.unique(y_resampled, return_counts=True)}")

Before --> y (unique): (array([0, 1]), array([158, 204]))
After  --> y (unique): (array([0, 1]), array([158, 149]))


In [108]:
model.fit(X_resampled, y_resampled)

y_train_pred = model.predict(X_train)

print("Train Classification report\n",classification_report(y_train, y_train_pred))

y_test_pred = model.predict(X_test)

print("Test Classification report\n",classification_report(y_test, y_test_pred))

Train Classification report
               precision    recall  f1-score   support

           0       0.69      0.94      0.80       158
           1       0.93      0.68      0.78       204

    accuracy                           0.79       362
   macro avg       0.81      0.81      0.79       362
weighted avg       0.83      0.79      0.79       362

Test Classification report
               precision    recall  f1-score   support

           0       0.44      0.58      0.50        67
           1       0.59      0.45      0.51        89

    accuracy                           0.51       156
   macro avg       0.52      0.52      0.51       156
weighted avg       0.53      0.51      0.51       156



In [109]:
saveModel(model=model, n_features=X_train.shape[1], technique_name="tomek-links")

### Cluster Centroids

In [110]:
from imblearn.under_sampling import ClusterCentroids

cc = ClusterCentroids(random_state=42)
X_resampled, y_resampled = cc.fit_resample(X, y)

print(f"Before --> y (unique): {np.unique(y_train, return_counts=True)}\nAfter  --> y (unique): {np.unique(y_resampled, return_counts=True)}")

Before --> y (unique): (array([0, 1]), array([158, 204]))
After  --> y (unique): (array([0, 1]), array([225, 225]))


In [111]:
model.fit(X_resampled, y_resampled)

y_train_pred = model.predict(X_train)

print("Train Classification report\n",classification_report(y_train, y_train_pred))

y_test_pred = model.predict(X_test)

print("Test Classification report\n",classification_report(y_test, y_test_pred))

Train Classification report
               precision    recall  f1-score   support

           0       0.64      0.86      0.73       158
           1       0.85      0.62      0.72       204

    accuracy                           0.73       362
   macro avg       0.75      0.74      0.73       362
weighted avg       0.76      0.73      0.73       362

Test Classification report
               precision    recall  f1-score   support

           0       0.73      0.79      0.76        67
           1       0.83      0.78      0.80        89

    accuracy                           0.78       156
   macro avg       0.78      0.78      0.78       156
weighted avg       0.79      0.78      0.78       156



In [112]:
saveModel(model=model, n_features=X_train.shape[1], technique_name="cluster-centroids")

### SMOTE + Tomek Links

In [113]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_resampled, y_resampled = smt.fit_resample(X_train, y_train)

print(f"Before --> y (unique): {np.unique(y_train, return_counts=True)}\nAfter  --> y (unique): {np.unique(y_resampled, return_counts=True)}")

Before --> y (unique): (array([0, 1]), array([158, 204]))
After  --> y (unique): (array([0, 1]), array([159, 159]))


In [114]:
model.fit(X_resampled, y_resampled)

y_train_pred = model.predict(X_train)

print("Train Classification report\n",classification_report(y_train, y_train_pred))

y_test_pred = model.predict(X_test)

print("Test Classification report\n",classification_report(y_test, y_test_pred))

Train Classification report
               precision    recall  f1-score   support

           0       0.74      0.73      0.73       158
           1       0.79      0.80      0.80       204

    accuracy                           0.77       362
   macro avg       0.77      0.77      0.77       362
weighted avg       0.77      0.77      0.77       362

Test Classification report
               precision    recall  f1-score   support

           0       0.42      0.40      0.41        67
           1       0.57      0.58      0.57        89

    accuracy                           0.51       156
   macro avg       0.49      0.49      0.49       156
weighted avg       0.50      0.51      0.50       156



In [115]:
saveModel(model=model, n_features=X_train.shape[1], technique_name="smote-tomeklinks")

### SMOTE + ENN

In [116]:
from imblearn.combine import SMOTEENN

sme = SMOTEENN(random_state=42)
X_resampled, y_resampled = sme.fit_resample(X_train, y_train)

print(f"Before --> y (unique): {np.unique(y_train, return_counts=True)}\nAfter  --> y (unique): {np.unique(y_resampled, return_counts=True)}")

Before --> y (unique): (array([0, 1]), array([158, 204]))
After  --> y (unique): (array([0, 1]), array([37, 24]))


In [117]:
model.fit(X_resampled, y_resampled)

y_train_pred = model.predict(X_train)

print("Train Classification report\n",classification_report(y_train, y_train_pred))

y_test_pred = model.predict(X_test)

print("Test Classification report\n",classification_report(y_test, y_test_pred))

Train Classification report
               precision    recall  f1-score   support

           0       0.46      0.76      0.58       158
           1       0.63      0.32      0.42       204

    accuracy                           0.51       362
   macro avg       0.55      0.54      0.50       362
weighted avg       0.56      0.51      0.49       362

Test Classification report
               precision    recall  f1-score   support

           0       0.46      0.54      0.49        67
           1       0.60      0.52      0.55        89

    accuracy                           0.53       156
   macro avg       0.53      0.53      0.52       156
weighted avg       0.54      0.53      0.53       156



In [118]:
saveModel(model=model, n_features=X_train.shape[1], technique_name="smote-enn")